In [1]:
import pandas as pd
import numpy as np
from astropy.io import fits
from astropy.table import Table, vstack
from astropy.stats import sigma_clip
import time
from scipy.stats import binned_statistic
import matplotlib.pyplot as plt
from astropy.stats import sigma_clip
from pathlib import Path


# Define the Information you Want to Take from the FastSpecFit Catalog #

In [2]:
# ---------------------------------------------
# 2. Read only the needed columns from FASTSPEC
# ---------------------------------------------
desired_host_cols = [
    "TARGETID",
    "HEALPIX", 
    
    "VDISP",
    "VDISP_IVAR",
    
    "LOGMSTAR", 
    # "LOGMSTAR_IVAR",
    
    "SFR",
    # "SFR_IVAR",
    
    "AGE",
    # "AGE_IVAR", 
    
    "DN4000",
    "DN4000_IVAR",

    
    "ABSMAG01_SDSS_G",
    # "ABSMAG01_IVAR_SDSS_G",

    "ZZSUN",
    # "ZZSUN_IVAR",
    
    "ABSMAG01_SDSS_R",
    # "ABSMAG01_IVAR_SDSS_R",

    # "TAUV",
    # "TAUV_IVAR",

    # "ABSMAG01_SDSS_U",
    # "ABSMAG01_IVAR_SDSS_U"
    
]


# FastSpecFit File Content #

Description of files you can query from the catalogue. More details on descriptions can be found here:

https://fastspecfit.readthedocs.io/en/2.5.2/fastspec.html

Note this is an older version of the FastSpecFit catalogue, but it is the only version available to us due to hte public release nature of the data. 


In [1]:
from astropy.io import fits

# Path to the FastSpecFit file
fastspecfit_path = "/global/cfs/cdirs/desi/public/dr1/vac/dr1/fastspecfit/iron/v2.1/catalogs/fastspec-iron-main-bright.fits"

# Open and inspect the FITS file
with fits.open(fastspecfit_path) as hdul:
    for i, hdu in enumerate(hdul):
        if isinstance(hdu, fits.BinTableHDU):
            columns = hdu.columns.names
            print(f"HDU {i}: {hdu.name}")
            print("Columns:")
            print(columns)
            print(len(columns))

HDU 1: FASTSPEC
Columns:
['TARGETID', 'SURVEY', 'PROGRAM', 'HEALPIX', 'Z', 'COEFF', 'RCHI2', 'RCHI2_CONT', 'RCHI2_PHOT', 'SNR_B', 'SNR_R', 'SNR_Z', 'SMOOTHCORR_B', 'SMOOTHCORR_R', 'SMOOTHCORR_Z', 'VDISP', 'VDISP_IVAR', 'AV', 'AGE', 'ZZSUN', 'LOGMSTAR', 'SFR', 'DN4000', 'DN4000_OBS', 'DN4000_IVAR', 'DN4000_MODEL', 'FLUX_SYNTH_G', 'FLUX_SYNTH_R', 'FLUX_SYNTH_Z', 'FLUX_SYNTH_SPECMODEL_G', 'FLUX_SYNTH_SPECMODEL_R', 'FLUX_SYNTH_SPECMODEL_Z', 'FLUX_SYNTH_PHOTMODEL_G', 'FLUX_SYNTH_PHOTMODEL_R', 'FLUX_SYNTH_PHOTMODEL_Z', 'FLUX_SYNTH_PHOTMODEL_W1', 'FLUX_SYNTH_PHOTMODEL_W2', 'FLUX_SYNTH_PHOTMODEL_W3', 'FLUX_SYNTH_PHOTMODEL_W4', 'ABSMAG10_DECAM_G', 'ABSMAG10_IVAR_DECAM_G', 'KCORR10_DECAM_G', 'ABSMAG10_DECAM_R', 'ABSMAG10_IVAR_DECAM_R', 'KCORR10_DECAM_R', 'ABSMAG10_DECAM_Z', 'ABSMAG10_IVAR_DECAM_Z', 'KCORR10_DECAM_Z', 'ABSMAG00_U', 'ABSMAG00_IVAR_U', 'KCORR00_U', 'ABSMAG00_B', 'ABSMAG00_IVAR_B', 'KCORR00_B', 'ABSMAG00_V', 'ABSMAG00_IVAR_V', 'KCORR00_V', 'ABSMAG01_SDSS_U', 'ABSMAG01_IVAR_SDSS_U', 

In [4]:
with fits.open(fastspecfit_path) as hdul:
    hdul.info()

Filename: /global/cfs/cdirs/desi/public/dr1/vac/dr1/fastspecfit/iron/v2.1/catalogs/fastspec-iron-main-bright.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      55   ()      
  1  FASTSPEC      1 BinTableHDU   1826   6445926R x 907C   [K, 4A, 6A, J, D, 40E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, E, E, E, D, D, D, D, D, D, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, E, J, E, E, E, E, E, E, E, E, E, 

# BRIGHT GALAXIES BULK FILE #

So in the following strategy we will try to get the data that we need by reading TargetID's from the following .fits files:

`desi_galaxy_metadata.fits` (which means that "bright" is implied) and `desi_galaxy_metadata_dark.fits` (where "dark is in the name)

And search them for their TARGETID's. This is really good because these .fits files have already been filtered by "Galaxies". 

We will take relevant galactic properties from the ` 📁 HDU 1: FASTSPEC` and save them in another fits file.

This strategy is mainly emloyed because when trying to serach all of `HDU 1: FASTSPEC` from just a .CSV file of `ZTF_snia_match_DESI_hostgal_Z_filtered_mu_residuals.csv` and found that this crashed the kernel. 

So it would be easier to just get a .fits file of ALL releavan desi galaxy metadata, save it, and then use the .csv to search for all these types of things. 

NOTE: we are only searching through the `Bright` Galaxies bulk file, because we decided not use the dark galaxies survey due to maintaining uniform uncertainty in the galaxy redshift. 

In [5]:
# ---------------------------------------------
# Paths
# ---------------------------------------------
#Path to metadata. 
fastspec_path = "/global/cfs/cdirs/desi/public/dr1/vac/dr1/fastspecfit/iron/v2.1/catalogs/fastspec-iron-main-bright.fits"
output_host_path = "../../myc21_first_paper_LARGE_DATA/desi_galaxy_fastspec_hostprops_bright.fits"

start = time.time()

print("Requested columns from FASTSPEC:", desired_host_cols)

# Read from the FASTSPEC HDU, only these columns
fastspec = Table.read(
    fastspec_path,
    hdu="FASTSPEC",       # HDU that contains host properties.
)

# Add this line to keep only the host properties you want
# Drops other collumns.
fastspec = fastspec[desired_host_cols]

fastspec.write(output_host_path, overwrite=True)

end = time.time()

print(f"Total time: {end - start:.2f} seconds")

# ---------------------------------------------
# 5. Quick sanity check of the output file
# ---------------------------------------------
print("\n Inspecting saved host-properties file:")
host_check = Table.read(output_host_path, hdu=1)
print("Output columns:", host_check.colnames)
print("Output length:", len(host_check))
print("First 5 rows:")
print(host_check[:5])

Requested columns from FASTSPEC: ['TARGETID', 'HEALPIX', 'VDISP', 'VDISP_IVAR', 'LOGMSTAR', 'SFR', 'AGE', 'DN4000', 'DN4000_IVAR', 'ABSMAG01_SDSS_G', 'ZZSUN', 'ABSMAG01_SDSS_R']
Total time: 78.18 seconds

 Inspecting saved host-properties file:
Output columns: ['TARGETID', 'HEALPIX', 'VDISP', 'VDISP_IVAR', 'LOGMSTAR', 'SFR', 'AGE', 'DN4000', 'DN4000_IVAR', 'ABSMAG01_SDSS_G', 'ZZSUN', 'ABSMAG01_SDSS_R']
Output length: 6445926
First 5 rows:
    TARGETID     HEALPIX VDISP ... ABSMAG01_SDSS_G ZZSUN ABSMAG01_SDSS_R
---------------- ------- ----- ... --------------- ----- ---------------
2368701667999750   10181 125.0 ...      -20.444702   0.0      -20.315044
2368786908839943    7420 125.0 ...      -22.611683   0.0      -22.759594
2368789538668549    7422 125.0 ...      -21.445084   0.0      -21.561594
2376125435084800   48990 125.0 ...      -25.705568   0.0      -27.367401
2376164282728448   40901 125.0 ...      -25.971771   0.0      -27.638779


# Host Galaxy Parameters to the .csv with SNIa Resduals #


In [6]:
# ------------------------------------------------------------------
# File paths – tweak these if your layout is different
# ------------------------------------------------------------------

csv_path = "data/ZTF_DESI_matched_lc_cuts_z_cuts.csv"
bright_fits_path = "../../myc21_first_paper_LARGE_DATA/desi_galaxy_fastspec_hostprops_bright.fits"

# ------------------------------------------------------------------
# 1) Read the SN Ia + residuals CSV
# ------------------------------------------------------------------
sne_df = pd.read_csv(csv_path)

# ------------------------------------------------------------------
# 2) Read both FASTSPEC host-property FITS files and stack them
# ------------------------------------------------------------------
bright_tab = Table.read(bright_fits_path)

# ------------------------------------------------------------------
# 3) Convert host table to pandas and clean up TARGETID
# ------------------------------------------------------------------
host_df = bright_tab.to_pandas()
host_df = host_df.add_prefix("DESI_FASTSPEC_")


# ------------------------------------------------------------------
# 4) Merge SN+residuals with host galaxy properties on TARGETID
# ------------------------------------------------------------------

merged = sne_df.merge(host_df, left_on="DESI_METADATA_TARGETID", right_on="DESI_FASTSPEC_TARGETID", how="left")

# ------------------------------------------------------------------
# 5) Save merged table
# ------------------------------------------------------------------
merged.to_csv("data/ZTF_DESI_matched_lc_cuts_z_cuts_hostprop.csv", index=False)

print(merged.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 539 entries, 0 to 538
Data columns (total 65 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   ZTF_Unnamed: 0                    539 non-null    int64  
 1   ztfname                           539 non-null    object 
 2   ZTF_redshift                      539 non-null    float64
 3   ZTF_redshift_err                  539 non-null    float64
 4   ZTF_source                        539 non-null    object 
 5   ZTF_t0                            539 non-null    float64
 6   ZTF_x0                            539 non-null    float64
 7   ZTF_x1                            539 non-null    float64
 8   ZTF_c                             539 non-null    float64
 9   ZTF_t0_err                        539 non-null    float64
 10  ZTF_x0_err                        539 non-null    float64
 11  ZTF_x1_err                        539 non-null    float64
 12  ZTF_c_er